# Feature Selection — Selección de features

**Objetivo:** Encontrar las mejores features usando dos enfoques y comparar resultados.

**Estrategia:**
- **Ruta A:** Partir de tests estadísticos → filtrar → RF importance
- **Ruta B:** Partir de TODAS las features → RF importance directo
- **Comparar** para diseñar los experimentos de entrenamiento

---
## 1. Setup

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

---
## 2. Carga de datos

In [12]:
df = pd.read_csv('../data/processed/application_train_preprocessed.csv')
tests = pd.read_csv('../data/processed/statistical_tests_summary.csv')

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

all_features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Total features: {len(all_features)}')
print(f'Tests evaluados: {tests.shape[0]}')

Dataset: 307,511 filas x 219 columnas
Total features: 217
Tests evaluados: 217


---
## RUTA A — Desde tests estadísticos

Filramos features con evidencia (p < 0.05 Y efecto ≥ 0.1), luego verificamos multicolinealidad, y finalmente evaluamos con Random Forest.

### A.1 Filtrar por evidencia estadística

In [13]:
util_features = tests[tests['Util_practica'] == True]['Feature'].tolist()

print(f'Features con evidencia: {len(util_features)}')
print()
for f in util_features:
    row = tests[tests['Feature'] == f].iloc[0]
    print(f'  {f:45s} {row["Tipo"]:12s} efecto={row["Effect_size"]:.3f}')

Features con evidencia: 16

  DAYS_BIRTH                                    Numérica     efecto=0.166
  bureau_avg_days_credit                        Numérica     efecto=0.180
  DAYS_CREDIT_UPDATE_mean                       Numérica     efecto=0.157
  CREDIT_ACTIVE_Active_pct                      Numérica     efecto=0.150
  bureau_debt_ratio                             Numérica     efecto=0.160
  bureau_active_rate                            Numérica     efecto=0.150
  prev_reject_rate                              Numérica     efecto=0.120
  NAME_CONTRACT_STATUS_Refused_pct              Numérica     efecto=0.120
  bureau_newest_credit_days                     Numérica     efecto=0.142
  credit_approval_ratio                         Numérica     efecto=0.138
  bureau_oldest_credit_days                     Numérica     efecto=0.137
  prev_avg_decision_days                        Numérica     efecto=0.115
  DAYS_LAST_PHONE_CHANGE                        Numérica     efecto=0.114
  DAYS_ID_

### A.2 Verificar multicolinealidad

In [14]:
# Matriz de correlación
corr_matrix = df[util_features].corr().abs()

# Encontrar pares altamente correlacionados
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            high_corr.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlacion': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr).sort_values('Correlacion', ascending=False)

print(f'Pares con correlación > 0.9: {len(high_corr_df)}')
high_corr_df

Pares con correlación > 0.9: 2


,Feature_1,Feature_2,Correlacion
0,CREDIT_ACTIVE_Active_pct,bureau_active_rate,1.0
1,prev_reject_rate,NAME_CONTRACT_STATUS_Refused_pct,1.0


In [15]:
# Eliminar una de cada par
corr_with_target = df.drop(columns=[ID_COL]).corr(method='spearman')[TARGET_COL]

cols_to_remove = set()
for _, row in high_corr_df.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    if f1 in cols_to_remove or f2 in cols_to_remove:
        continue
    
    corr_f1 = abs(corr_with_target.get(f1, 0))
    corr_f2 = abs(corr_with_target.get(f2, 0))
    
    remove = f2 if corr_f1 >= corr_f2 else f1
    keep = f1 if corr_f1 >= corr_f2 else f2
    
    cols_to_remove.add(remove)
    print(f'  Eliminar: {remove:40s} (corr_target={corr_with_target.get(remove, 0):.4f})')
    print(f'  Mantener: {keep:40s} (corr_target={corr_with_target.get(keep, 0):.4f})')
    print()

route_a_features = [f for f in util_features if f not in cols_to_remove]

print(f'Ruta A después de multicolinealidad: {len(route_a_features)} features')

  Eliminar: bureau_active_rate                       (corr_target=0.0711)
  Mantener: CREDIT_ACTIVE_Active_pct                 (corr_target=0.0711)

  Eliminar: NAME_CONTRACT_STATUS_Refused_pct         (corr_target=0.0677)
  Mantener: prev_reject_rate                         (corr_target=0.0677)

Ruta A después de multicolinealidad: 14 features


### A.3 Random Forest en features filtradas

In [18]:
X_a = df[route_a_features]
y = df[TARGET_COL]

rf_a = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_a.fit(X_a, y)

importances_a = pd.Series(rf_a.feature_importances_, index=X_a.columns)
importances_a = importances_a.sort_values(ascending=False)

# Filtrar por importancia mínima
threshold = 0.001
route_a_final = [f for f in importances_a.index if importances_a[f] > threshold]

print('RUTA A por Random Forest')
print('=' * 60)
for i, (col, val) in enumerate(importances_a.head(15).items(), 1):
    print(f'  {i:2d}. {col:40s} {val:.4f}')

print(f'\nRuta A final: {len(route_a_final)} features')

RUTA A por Random Forest
   1. DAYS_BIRTH                               0.1218
   2. credit_approval_ratio                    0.1173
   3. bureau_avg_days_credit                   0.1073
   4. prev_reject_rate                         0.0995
   5. bureau_debt_ratio                        0.0801
   6. bureau_newest_credit_days                0.0726
   7. DAYS_LAST_PHONE_CHANGE                   0.0640
   8. DAYS_CREDIT_UPDATE_mean                  0.0616
   9. prev_avg_decision_days                   0.0491
  10. CREDIT_ACTIVE_Active_pct                 0.0489
  11. bureau_avg_debt                          0.0468
  12. DAYS_ID_PUBLISH                          0.0463
  13. bureau_oldest_credit_days                0.0448
  14. prev_oldest_decision_days                0.0401

Ruta A final: 14 features


---
## RUTA B — Desde todas las features

Entrenamos Random Forest con TODAS las features y vemos cuáles son importantes.

### B.1 Random Forest en todas las features

In [19]:
X_b = df[all_features]
y = df[TARGET_COL]

print(f'Features para Ruta B: {X_b.shape[1]}')
print(f'Muestras: {X_b.shape[0]:,}')

Features para Ruta B: 217
Muestras: 307,511


In [21]:
rf_b = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_b.fit(X_b, y)

importances_b = pd.Series(rf_b.feature_importances_, index=X_b.columns)
importances_b = importances_b.sort_values(ascending=False)

# Filtrar por importancia mínima
route_b_final = [f for f in importances_b.index if importances_b[f] > threshold]

print('RUTA B — Top 15 por Random Forest y todas las features')
print('=' * 60)
for i, (col, val) in enumerate(importances_b.head(15).items(), 1):
    print(f'  {i:2d}. {col:40s} {val:.4f}')

print(f'\nRuta B final: {len(route_b_final)} features')

RUTA B — Top 15 por Random Forest y todas las features
   1. ext_source_mean                          0.1708
   2. ext_source_min                           0.1024
   3. ext_source_max                           0.0897
   4. EXT_SOURCE_2                             0.0640
   5. EXT_SOURCE_3                             0.0529
   6. EXT_SOURCE_1                             0.0218
   7. credit_approval_ratio                    0.0185
   8. goods_to_credit                          0.0173
   9. bureau_avg_days_credit                   0.0164
  10. ext_source_std                           0.0146
  11. bureau_debt_ratio                        0.0141
  12. age_years                                0.0137
  13. DAYS_BIRTH                               0.0121
  14. NAME_CONTRACT_STATUS_Refused_pct         0.0114
  15. NAME_EDUCATION_TYPE                      0.0112

Ruta B final: 109 features


---
## COMPARACIÓN — Ambas rutas

In [22]:
# Comparar rutas
set_a = set(route_a_final)
set_b = set(route_b_final)

coinciden = set_a & set_b
solo_a = set_a - set_b
solo_b = set_b - set_a

print('COMPARACIÓN DE RUTAS')
print('=' * 60)
print(f'Ruta A (tests → RF):  {len(route_a_final)} features')
print(f'Ruta B (RF directo):  {len(route_b_final)} features')
print()
print(f'Coinciden:            {len(coinciden)} features')
print(f'Solo en Ruta A:       {len(solo_a)} features')
print(f'Solo en Ruta B:       {len(solo_b)} features')

COMPARACIÓN DE RUTAS
Ruta A (tests → RF):  14 features
Ruta B (RF directo):  109 features

Coinciden:            14 features
Solo en Ruta A:       0 features
Solo en Ruta B:       95 features


In [23]:
# Features que coinciden
print('FEATURES QUE COINCIDEN (ambas rutas)')
print('=' * 60)
for f in sorted(coinciden):
    imp_a = importances_a.get(f, 0)
    imp_b = importances_b.get(f, 0)
    print(f'  {f:40s} RF_A={imp_a:.4f}  RF_B={imp_b:.4f}')

FEATURES QUE COINCIDEN (ambas rutas)
  CREDIT_ACTIVE_Active_pct                 RF_A=0.0489  RF_B=0.0084
  DAYS_BIRTH                               RF_A=0.1218  RF_B=0.0121
  DAYS_CREDIT_UPDATE_mean                  RF_A=0.0616  RF_B=0.0096
  DAYS_ID_PUBLISH                          RF_A=0.0463  RF_B=0.0048
  DAYS_LAST_PHONE_CHANGE                   RF_A=0.0640  RF_B=0.0060
  bureau_avg_days_credit                   RF_A=0.1073  RF_B=0.0164
  bureau_avg_debt                          RF_A=0.0468  RF_B=0.0063
  bureau_debt_ratio                        RF_A=0.0801  RF_B=0.0141
  bureau_newest_credit_days                RF_A=0.0726  RF_B=0.0104
  bureau_oldest_credit_days                RF_A=0.0448  RF_B=0.0065
  credit_approval_ratio                    RF_A=0.1173  RF_B=0.0185
  prev_avg_decision_days                   RF_A=0.0491  RF_B=0.0045
  prev_oldest_decision_days                RF_A=0.0401  RF_B=0.0053
  prev_reject_rate                         RF_A=0.0995  RF_B=0.0092


In [24]:
# Features solo en Ruta A
if solo_a:
    print('FEATURES SOLO EN RUTA A (tests estadísticos)')
    print('=' * 60)
    for f in sorted(solo_a):
        imp_a = importances_a.get(f, 0)
        print(f'  {f:40s} RF_A={imp_a:.4f}')
    print()
    print('Nota: Estas features pasaron los tests pero RF las considera')
    print('menos importantes. Posibrelmente capturan relaciones lineales.')
else:
    print('No hay features solo en Ruta A.')

No hay features solo en Ruta A.


In [25]:
# Features solo en Ruta B
if solo_b:
    print('FEATURES SOLO EN RUTA B (RF directo)')
    print('=' * 60)
    for f in sorted(solo_b):
        imp_b = importances_b.get(f, 0)
        print(f'  {f:40s} RF_B={imp_b:.4f}')
    print()
    print('Nota: Estas features no pasaron los tests pero RF las considera')
    print('importantes. Posiblemente capturan relaciones no lineales.')
else:
    print('No hay features solo en Ruta B.')

FEATURES SOLO EN RUTA B (RF directo)
  AMT_ANNUITY                              RF_B=0.0056
  AMT_ANNUITY_max_x                        RF_B=0.0011
  AMT_ANNUITY_max_y                        RF_B=0.0035
  AMT_ANNUITY_mean_x                       RF_B=0.0012
  AMT_ANNUITY_mean_y                       RF_B=0.0037
  AMT_CREDIT                               RF_B=0.0055
  AMT_CREDIT_max                           RF_B=0.0030
  AMT_DOWN_PAYMENT_mean                    RF_B=0.0048
  AMT_DOWN_PAYMENT_sum                     RF_B=0.0092
  AMT_GOODS_PRICE                          RF_B=0.0068
  AMT_INCOME_TOTAL                         RF_B=0.0023
  APARTMENTS_AVG                           RF_B=0.0018
  APARTMENTS_MEDI                          RF_B=0.0016
  APARTMENTS_MODE                          RF_B=0.0018
  BASEMENTAREA_AVG                         RF_B=0.0016
  BASEMENTAREA_MEDI                        RF_B=0.0016
  BASEMENTAREA_MODE                        RF_B=0.0015
  CNT_PAYMENT_max           

---
## Diseño de experimentos

Basado en la comparación, podemos diseñar los configs YAML:

In [26]:
# Crear sets de features para experimentos
feature_sets = {
    'route_a': route_a_final,
    'route_b': route_b_final,
    'intersection': sorted(coinciden),
    'union': sorted(set_a | set_b)
}

print('SETS DE FEATURES PARA EXPERIMENTOS')
print('=' * 60)
for name, features in feature_sets.items():
    print(f'  {name:20s}: {len(features):3d} features')

SETS DE FEATURES PARA EXPERIMENTOS
  route_a             :  14 features
  route_b             : 109 features
  intersection        :  14 features
  union               : 109 features


---
## Guardar resultados

In [27]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar cada set de features
for name, features in feature_sets.items():
    filepath = f'../data/processed/features_{name}.txt'
    with open(filepath, 'w') as f:
        for feat in features:
            f.write(feat + '\n')
    print(f'Guardado: {filepath} ({len(features)} features)')

# Guardar dataset con la ruta ganadora
df_final = df[[ID_COL, TARGET_COL] + route_a_final]
df_final.to_csv('../data/processed/application_train_selected.csv', index=False)
print(f'\nDataset final (Ruta A): {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas')

Guardado: ../data/processed/features_route_a.txt (14 features)
Guardado: ../data/processed/features_route_b.txt (109 features)
Guardado: ../data/processed/features_intersection.txt (14 features)
Guardado: ../data/processed/features_union.txt (109 features)

Dataset final (Ruta A): 307,511 filas x 16 columnas


---
## Conclusiones para experimentos

### Hallazgos principales:

#### 1. Ruta A vs Ruta B: Diferencias fundamentales

| Ruta | Objetivo |
|------|----------|
| A (Tests → RF) con 14 features  | **individualmente predictivas** |
| B (RF directo) con 109 | **más útiles en conjunto**|


#### 2. Features exclusivas de Ruta A: 0

Ninguna, porque todas las features que pasaron los tests estadísticos (p < 0.05, |r| ≥ 0.1) **también** son importantes para RF.

#### 3. Features exclusivas de Ruta B: 95 features adicionales

RF encontró 95 features que **no pasaron los tests univariados** pero son útiles en combinación. 


**Aquí es donde se ve que los tests estadísticos tienen limitaciones y no se pueden descartar las features solo porque no salen en un test.**

#### 4. Intersección: Las 14 más robustas

Las features que aparecen en **ambas rutas** son las más confiables:
- Tienen **evidencia individual** (pasaron tests estadísticos)
- Son **útiles en contexto** (RF las valora)
- Capturan **tanto relaciones lineales como no lineales**

### Univariado vs Multivariado

Los tests estadísticos (Mann-Whitney, Spearman) son **univariados**: evalúan cada feature **individualmente** contra el target.

Random Forest importance es **multivariado**: considera las features **juntas** y mide cuánto contribuyen al modelo.

**Implicación práctica:**
- Tests estadísticos: Buenos para **modelos lineales** y necesitan relaciones individuales fuertes.
- RF importance: Buenos para **modelos no lineales** y capturan interacciones
- **Ambos juntos**: Visión completa del potencial de cada feature

### Experimentos:

- **exp001:** Ruta A (14 features) — Set conservador, bajo riesgo, para modelos lineales
- **exp002:** Ruta B (109 features) — Set exploratorio, para modelos no lineales
- **exp003:** Intersección (14 features) — Mismo que Ruta A (ya que son iguales)
- **exp004:** Top 30 de Ruta B — Balance entre complejidad y performance
